In [ ]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:

!pip install sacrebleu
import sacrebleu

# References are a list of lists (for multi-reference support)
references = [["The cat is on the mat", "There is a cat on the mat"]]
# Hypotheses are a list
hypotheses = ["The cat is on the mat"]

# Compute BLEU score
bleu = sacrebleu.corpus_bleu(hypotheses, references)
print(bleu.score)


100.00000000000004


In [ ]:
!pip install transformers[sentencepiece] datasets sacrebleu rouge_score py7zr -q

In [ ]:
!pip install matplotlib

In [ ]:
%pip install matplotlib
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from tqdm import tqdm
import torch
nltk.download("punkt")

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to C:\Users\Harsha
[nltk_data]     vardhan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [ ]:

model_ckpt = "google/pegasus-cnn_dailymail"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt).to(device)


Loading weights: 100%|██████████| 680/680 [00:01<00:00, 583.90it/s, Materializing param=model.shared.weight]                                   
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:th

In [ ]:
import urllib.request
import zipfile
import os

# Download the file
url = "https://github.com/entbappy/Branching-tutorial/raw/master/summarizer-data.zip"
urllib.request.urlretrieve(url, "summarizer-data.zip")

# Extract with renaming problematic files
with zipfile.ZipFile("summarizer-data.zip", 'r') as zip_ref:
    for member in zip_ref.namelist():
        # Skip problematic .arrow files or rename them
        if member.endswith('.arrow'):
            # Extract to a safe filename
            target = member.replace('data-00000-of-00001.arrow', 'data.arrow')
            target = os.path.normpath(target)
            dirname = os.path.dirname(target)
            if dirname:
                os.makedirs(dirname, exist_ok=True)
            with zip_ref.open(member) as source:
                with open(target, 'wb') as target_file:
                    target_file


In [ ]:
dataset_samsum = load_from_disk('samsum_dataset')
dataset_samsum

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14732
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
})

In [ ]:
split_lengths = [len(dataset_samsum[split])for split in dataset_samsum]
print (f"split lengths : {dataset_samsum['train'].column_names}")
print("\nDialogue:")

print(dataset_samsum["test"][1]["dialogue"])
print("\nSummary:")

print(dataset_samsum["test"][1]["summary"])

split lengths : ['id', 'dialogue', 'summary']

Dialogue:
Eric: MACHINE!
Rob: That's so gr8!
Eric: I know! And shows how Americans see Russian ;)
Rob: And it's really funny!
Eric: I know! I especially like the train part!
Rob: Hahaha! No one talks to the machine like that!
Eric: Is this his only stand-up?
Rob: Idk. I'll check.
Eric: Sure.
Rob: Turns out no! There are some of his stand-ups on youtube.
Eric: Gr8! I'll watch them now!
Rob: Me too!
Eric: MACHINE!
Rob: MACHINE!
Eric: TTYL?
Rob: Sure :)

Summary:
Eric and Rob are going to watch a stand-up on youtube.


In [ ]:
def convert_examples_to_features(example_batch):
  input_encodings=tokenizer(example_batch['dialogue'],max_length=1024,truncation=True)
  target_encodings=tokenizer(example_batch['summary'],max_length=128,truncation=True)
  return {
    'input_ids':input_encodings['input_ids'],
    'attention_mask':input_encodings['attention_mask'],
    'labels':target_encodings['input_ids']
  }

In [ ]:

dataset_samsum_pt=dataset_samsum.map(convert_examples_to_features,batched=True)

In [ ]:
!pip install accelerate>=1.1.0


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

from transformers import TrainingArguments

trainer_args = TrainingArguments(
    output_dir='pegasus-samsum',
    num_train_epochs=1,
    warmup_steps=500,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=1e6,
    gradient_accumulation_steps=16
)


c:\Users\Harsha vardhan\OneDrive\Desktop\TextSummarization-Project\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collector = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [ ]:
trainer=Trainer(model=model_pegasus,args=trainer_args,
    data_collator=seq2seq_data_collector,
                train_dataset=dataset_samsum_pt["test"],
                eval_dataset=dataset_samsum_pt["validation"])

: 

In [ ]:
trainer.train()

In [ ]:
!pip install evaluate

In [ ]:
import evaluate

rogue_names=['roge1','roge2','rougeL',"rougeLsum"]
rogue_metric=evaluate.load('rouge')

In [ ]:
# evaluation

def generate_batch_sized_chunks(list_of_elements, batch_size):
  """split the dataset into smaller batches that we can process simultaneously
  Yield successive batch-sized chunks from list_of_elements."""
  for i in range(0, len(list_of_elements), batch_size):
    yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer, batch_size=16, device=device, column_text="dialogue", column_summary="summary"):
  article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
  target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

  for article_batch, target_batch in tqdm(
      zip(article_batches, target_batches), total=len(article_batches)):

      inputs = tokenizer(article_batch, max_length=1024, truncation=True, padding="max_length", return_tensors="pt")

      summaries = model.generate(
          input_ids=inputs["input_ids"].to(device),
          attention_mask=inputs["attention_mask"].to(device),
          length_penalty=0.8, num_beams=8, max_length=128
      )

      # finally, we decode the generated texts,
      # replace the token, and add the decoded texts with the reference to the metrics.

      decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                             for s in summaries]

      # decoded_summaries = [d.replace("", " ") for d in decoded_summaries] # This line seems incorrect/unnecessary
      metric.add_batch(predictions=decoded_summaries, references=target_batch)

  #finally compute and return the ROGUE scores
  score = metric.compute()
  return score

In [ ]:
import evaluate

rogue_names=['rouge1','rouge2','rougeL',"rougeLsum"]
rogue_metric=evaluate.load('rouge')

score=calculate_metric_on_test_ds(
    dataset_samsum['test'][0:10], rogue_metric,trainer.model,tokenizer,batch_size=2,column_text='dialogue',column_summary='summary'

)

rouge_dict=dict((rn,score[rn]) for rn in rogue_names)
pd.DataFrame([rouge_dict],index=['pegasus'])

In [ ]:
## save model
model_pegasus.save_pretrained("pegasus-samsum-model")

In [ ]:
tokenizer.save_pretrained("tokenize")

In [ ]:
#load
tokenizer=AutoTokenizer.from_pretrained("/content/pegasus-samsum/checkpoint-52", local_files_only=True)

In [ ]:
# prediction

gen_kwargs={"length_penalty":0.8,"num_beams":8,"max_length":128}

sample_text=dataset_samsum["test"][0]["dialogue"]
reference=dataset_samsum["test"][0]["summary"]

# Load the locally saved model and tokenizer objects from the checkpoint
local_model = AutoModelForSeq2SeqLM.from_pretrained("/content/pegasus-samsum/checkpoint-52", local_files_only=True).to(device)
local_tokenizer = AutoTokenizer.from_pretrained("/content/pegasus-samsum/checkpoint-52", local_files_only=True)

# Tokenize the input text
inputs = local_tokenizer(sample_text, truncation=True, padding="longest", return_tensors="pt").to(device)

# Generate summary
summary_ids = local_model.generate(inputs["input_ids"], **gen_kwargs)
summary_text = local_tokenizer.decode(summary_ids[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

print("Dialogue:")
print(sample_text)

print("\nReference summary:")
print(reference)

print("\nModel Summary:")
print(summary_text)